# 🌗 burst_scan — the extended hours: do the pre- and post-market help catch runs at the open and the close?

A companion to `move_explorer.ipynb`. That notebook traces **runs** (swings of at least `MOVE_PCT` ended by a
`REVERSAL_PCT` pullback) and **confirmations** (the first close `CONFIRM_PCT` off a recent low / high — the moment a
live identifier could act) on regular-hours bars only. Two places are blind there:

* **The open.** 39% of all regular-hours runs start in the first half hour, half of them *on the very first bar* —
  the regular-hours view cannot see what happened before 9:30.
* **The close.** Runs that start late are cut off at 16:00, and whatever they do next is invisible.

This notebook loads names that genuinely trade outside regular hours and runs the same machinery on **the same
prints twice**: once with the pre- and post-market (04:00-20:00 New York) and once without (09:30-16:00). Any
difference is the extended hours alone.

| § | question |
|---|---|
| 0 | the knobs; load one panel, mask the no-trade minutes, split it into the two views |
| 1 | **how tradable is each hour?** — prints, dollars and bar ranges by clock time |
| 2 | **where do runs really start?** — runs at the open that began in the pre-market; runs at the close that went on after it; after-hours runs |
| 3 | **the open, predicted at 9:30** — does the pre-market know whether the first hour moves, and which way, better than the gap and the previous days? |
| 4 | **confirmations around the clock** — the trade from each confirmation, by half hour, with and without the extended hours |
| 5 | **the close** — late confirmations cut at 16:00 or run on into the evening; the evening runs themselves |
| 6 | example days |
| 7 | checks and the scorecard |

**The picks** (hard-coded, change freely): MU, INTC, ORCL, HOOD, COIN, SOFI, IONQ, SMCI — every one prints in 74-92%
of pre-market minutes from 07:00 and trades $25-180M there a day. (GLW and ALB, two of the explorer's names, print in
fewer than 30%: their pre-market is mostly empty.) Training window only; the holdout (May-June 2026) stays unread.

**The data — two traps, both masked before anything else.**

* IBKR stores a minute with fewer than 100 shares as a flat copy of the previous close with volume 0 — about a third
  of all pre- and post-market minutes even on these names (`research.mask_placeholders`).
* **08:00 New York carries stale prints.** That is when FINRA's trade reporting opens, and overnight trades are
  reported then at their overnight prices. 82% of 08:00 minutes stick out more than 25 bp beyond the prints either
  side (0.2% at 07:59), fading out by about 08:08 (Fig 1b). Left in, they become the pre-market's highs and lows,
  run onsets and stop-outs, and they manufactured a striking −33 bp "pre-market reversal" (§4b). Masked:
  `LATE_REPORT_MIN` minutes from 08:00 (`research.mask_late_reports`).

In [ ]:
# If this import fails: select the "trading-simple" Jupyter kernel
# (see README "Quick start"), or run  uv sync --extra notebook
import importlib.util
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from trading_simple import backtest as bt, plots, strategies
from trading_simple.config import load_settings
from trading_simple.data import resample_panel
from trading_simple.db import make_engine

settings = load_settings()
engine = make_engine(settings)
# research.py holds every computation the five notebooks share; signals() never imports it.
_spec = importlib.util.spec_from_file_location(
    "burst_research", strategies.STRATEGIES_DIR / "burst_scan" / "research.py"
)
rs = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(rs)
strat, S = rs.STRAT, rs.S  # the loaded strategy and its module's functions
compose = S["compose"]
SIDES = ("long", "short")
COLOR = {"long": plots._LONG, "short": plots._SHORT}


def below(fig, y=-0.1):
    """Legend under the plot: the default top-right legend sits on the subplot titles."""
    fig.update_layout(
        legend={"orientation": "h", "x": 0, "xanchor": "left", "y": y, "yanchor": "top"}
    )
    return fig


from trading_simple.data import load_panel

# --- 0 · the knobs --------------------------------------------------------------------------------
SYMBOLS = ["MU", "INTC", "ORCL", "HOOD", "COIN", "SOFI", "IONQ", "SMCI"]  # all trade pre-market
MOVE_PCT = 0.02  # a run is at least this big ...
WINDOW_MIN = 240  # ... and its first MOVE_PCT came within this many minutes of its onset
REVERSAL_PCT = 0.01  # a pullback of this much ends a run
CONFIRM_PCT = 0.005  # the live trigger: a close this far off the recent low (high)
LOOKBACK_MIN = 120  # "recent" low / high for a confirmation
OPEN_MIN = 30  # a run or confirmation in the first N minutes of regular hours is "at the open"
CLOSE_MIN = 60  # ... in the last N minutes, "at the close"
RACE_MIN = 60  # §3: from the opening print, did price move MOVE_PCT either way within N minutes?
TREND_MIN = 30  # §3: the pre-market's closing stretch (pm_trend)
LATE_REPORT_MIN = (
    10  # minutes from 08:00 treated as no print (stale overnight trades reported late)
)
BAR_MIN = 5  # candle size in minutes (must divide 330 — 04:00 to 09:30 — so no candle
# straddles the open)
PROFILE_DAYS = 10  # the extended-volume baselines: this many previous days
ATR_DAYS = 14  # daily ATR smoothing, in regular sessions
START, END = "2025-06-18", "2026-04-30"  # the training window; the holdout stays unread
CARDS = 3  # §6: example days of each kind
SEED = 0

## 0 · One panel, two views

In [ ]:
raw = load_panel(engine, SYMBOLS, "1m", START, END, session="both", exchanges=settings.exchanges)
stored1, _ = rs.split_sessions(rs.mask_placeholders(raw))  # as stored (placeholders aside)
ext1, _ = rs.split_sessions(
    rs.mask_late_reports(stored1, minutes=LATE_REPORT_MIN)
)  # + 08:00 masked
ext = resample_panel(ext1, BAR_MIN) if BAR_MIN > 1 else ext1
ext, reg = rs.split_sessions(ext)  # the SAME prints, with and without
# the regular view must be exactly what every other notebook loads (session="regular")
reg_db = rs.mask_placeholders(
    load_panel(engine, SYMBOLS, "1m", START, END, session="regular", exchanges=settings.exchanges)
)
reg_db = resample_panel(reg_db, BAR_MIN) if BAR_MIN > 1 else reg_db
same_prints = reg_db.close.index.equals(reg.close.index) and all(
    np.allclose(getattr(reg_db, k).to_numpy(), getattr(reg, k).to_numpy(), equal_nan=True)
    for k in ("open_", "high", "low", "close", "volume")
)
tz = ext.tz
parts = rs.session_parts(ext)
mpb = S["_minutes_per_bar"](ext.bar)
loc = ext.close.index.tz_convert(tz)
clock = np.asarray(loc.hour * 60 + loc.minute)  # minutes after local midnight
day_x = np.asarray(loc.date)
codes_r, bar_no_r = S["_sessions"](reg.close.index, tz)
left_r = pd.Series(codes_r).groupby(codes_r).transform("size").to_numpy() - 1 - bar_no_r
fees, slip = bt.strategy_costs(strat, reg, settings)
RT = 2 * (fees + slip)  # one regular-hours round trip at the order size
grid_ok = bool((((clock - 570) % BAR_MIN) == 0).all())  # every candle on the 09:30 grid
masked = int(((raw.volume == 0) & raw.close.notna()).to_numpy().sum())
print(
    f"{len(SYMBOLS)} stocks · {len(set(day_x))} days · {ext.bar} bars: {len(ext.close):,} extended "
    f"({(parts == 'pre').sum():,} pre-market, {(parts == 'rth').sum():,} regular, "
    f"{(parts == 'post').sum():,} after-hours) · {masked:,} placeholder minutes masked · "
    f"regular round trip {1e4 * RT:.1f} bp"
)

## 1 · How tradable is each hour? (Fig 1)

Per half hour of the day, on the 1-minute bars: **top** the share of minutes with at least one trade, **middle** the
median dollars traded in a minute that traded (log scale), **bottom** the lag-1 autocorrelation of print-to-print
returns (one traded minute's close to the next, within the day). A trade price alternating between bid and ask
makes consecutive changes cancel, so the more negative it is, the more of each "move" is the spread. Outside
regular hours (unshaded) a minute trades tens of thousands of dollars, not millions, and the bounce is several
times stronger: trade prices there carry the spread in them, and the regular-hours round trip (the red line in
Fig 4) is a floor for the real cost.

In [ ]:
loc1 = ext1.close.index.tz_convert(tz)
half = ((loc1.hour * 60 + loc1.minute) // 30) * 30 / 60  # hours, e.g. 9.5 = 09:30
printed = ext1.close.notna()
dollars = (ext1.close * ext1.volume).where(printed)


def bounce(sym):
    """Lag-1 autocorrelation of print-to-print log returns, per half hour (within each day)."""
    c = ext1.close[sym].dropna()
    lc = c.index.tz_convert(tz)
    r = np.log(c).groupby(lc.date).diff()
    d = pd.DataFrame(
        {
            "r": r,
            "r1": r.groupby(lc.date).shift(1),
            "h": ((lc.hour * 60 + lc.minute) // 30) * 30 / 60,
        }
    ).dropna()
    return d.groupby("h")[["r", "r1"]].corr().xs("r", level=1)["r1"]


fig = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    subplot_titles=(
        "share of minutes with a trade",
        "median $ per traded minute",
        "print-to-print return autocorrelation (more negative = more bid-ask bounce)",
    ),
)
for k, sym in enumerate(SYMBOLS):
    kw = {
        "mode": "lines",
        "name": sym,
        "line": {"color": plots.SERIES[k % len(plots.SERIES)], "width": 1.5},
    }
    fig.add_trace(
        go.Scatter(x=sorted(set(half)), y=printed[sym].groupby(half).mean().to_numpy(), **kw),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=sorted(set(half)),
            y=dollars[sym].groupby(half).median().to_numpy(),
            showlegend=False,
            **kw,
        ),
        row=2,
        col=1,
    )
    ac = bounce(sym)
    fig.add_trace(go.Scatter(x=ac.index, y=ac.to_numpy(), showlegend=False, **kw), row=3, col=1)
for row in (1, 2, 3):  # shapes AFTER their subplot's traces
    fig.add_vrect(
        x0=9.5,
        x1=16,
        fillcolor=plots._GRID,
        opacity=0.5,
        line_width=0,
        layer="below",
        row=row,
        col=1,
    )
fig.update_yaxes(tickformat=".0%", row=1, col=1)
fig.update_yaxes(type="log", row=2, col=1)
fig.update_xaxes(title_text="New York time (shaded: regular hours)", dtick=1, row=3, col=1)
plots._base(fig, title="Fig 1 · how much trades, hour by hour", height=720)
below(fig, -0.08)
fig.show()

parts_1 = rs.session_parts(ext1)
n_days = len(set(day_x))
liq = pd.DataFrame(
    {
        "pre-market $M / day": dollars[parts_1 == "pre"].sum() / n_days / 1e6,
        "regular $M / day": dollars[parts_1 == "rth"].sum() / n_days / 1e6,
        "after-hours $M / day": dollars[parts_1 == "post"].sum() / n_days / 1e6,
        "pre-market minutes traded (07:00-09:30)": printed[
            (parts_1 == "pre") & (loc1.hour >= 7)
        ].mean(),
        "after-hours minutes traded (16:00-18:00)": printed[
            (parts_1 == "post") & (loc1.hour < 18)
        ].mean(),
    }
)
display(liq.round(3))

# Fig 1b — the 08:00 stale prints, as stored: a wick far beyond the prints either side of the minute
c_s = stored1.close
around_hi = np.fmax(c_s.ffill().shift(1), c_s.bfill().shift(-1))
around_lo = np.fmin(c_s.ffill().shift(1), c_s.bfill().shift(-1))
wick = (
    1e4
    * np.fmax(stored1.high - np.fmax(around_hi, c_s), np.fmin(around_lo, c_s) - stored1.low)
    / c_s
)
minute = np.asarray(loc1.hour * 60 + loc1.minute)
near = (minute >= 450) & (minute < 540)
share = (wick[near] > 25).where(c_s[near].notna()).groupby(minute[near]).mean().mean(axis=1)
fig = go.Figure(
    go.Bar(
        x=[f"{m // 60:02d}:{m % 60:02d}" for m in share.index],
        y=share.to_numpy(),
        marker_color=[
            plots._LOSS if 480 <= m < 480 + LATE_REPORT_MIN else plots._MUTED for m in share.index
        ],
        name="as stored",
    )
)
plots._base(
    fig,
    title="Fig 1b · share of traded minutes with a wick > 25 bp beyond the prints either side "
    "(as stored; red = masked)",
    height=340,
)
fig.update_yaxes(tickformat=".0%")
fig.show()

## 2 · Where do runs really start? (Fig 2)

Runs are traced on each view: per regular session (09:30-16:00) and per whole local day (04:00-20:00). A run
at the open in the regular view is then looked up in the extended view — the extended run of the same name and
direction that contains its onset (`research.match_runs`) — to see whether it really began before 9:30, and how much of
it was already done by then. Late runs are followed past 16:00 the same way.

**Fig 2a** onsets by half hour in each view (the 04:00 bar collects runs whose "onset" is just the day's first
print: the move since 20:00 the evening before is not in the data, as 09:30 is in the regular view). **Fig 2b** for the runs that start at the open in the regular view:
when did the same run start in the extended view? **Fig 2c** the after-hours runs, against that evening's volume.

In [ ]:
kw_runs = dict(
    reversal_pct=REVERSAL_PCT, min_run_pct=MOVE_PCT, max_run_min=WINDOW_MIN, confirm_pct=CONFIRM_PCT
)
runs_x = rs.swing_runs(ext, **kw_runs)
runs_r = rs.swing_runs(reg, **kw_runs)
runs_x["part"] = parts[runs_x["onset"]]
runs_x["hour"] = clock[runs_x["onset"]] / 60
clock_r = np.asarray(
    reg.close.index.tz_convert(tz).hour * 60 + reg.close.index.tz_convert(tz).minute
)
runs_r["hour"] = clock_r[runs_r["onset"]] / 60
runs_r["at_open"] = bar_no_r[runs_r["onset"]] * mpb < OPEN_MIN
runs_r["at_close"] = left_r[runs_r["onset"]] * mpb < CLOSE_MIN
n_names_days = len(SYMBOLS) * len(set(day_x))
counts = pd.DataFrame(
    {
        "regular view": [0, len(runs_r), 0],
        "extended view": [int((runs_x["part"] == p).sum()) for p in ("pre", "rth", "post")],
    },
    index=["pre-market onset", "regular-hours onset", "after-hours onset"],
)
counts.loc["per name and day"] = counts.sum() / n_names_days
display(counts.round(2))

# --- the open: where did the regular view's open runs begin?
op = rs.match_runs(
    runs_r[runs_r["at_open"]].reset_index(drop=True), reg.close.index, runs_x, ext.close.index
)
op["found"] = op["x_onset_ts"].notna()
op["x_hour"] = (
    op["x_onset_ts"].dt.tz_convert(tz).dt.hour + op["x_onset_ts"].dt.tz_convert(tz).dt.minute / 60
)
op["began_pre"] = op["x_hour"] < 9.5
op["first_bar"] = bar_no_r[op["onset"]] == 0
up_ = op["direction"] > 0
op["done_by_open"] = np.where(up_, op["base"] / op["x_base"] - 1, 1 - op["base"] / op["x_base"])
f_ = op[op["found"]]
open_facts = {
    "open runs (share of all regular-hours runs)": len(op) / len(runs_r),
    "... whose regular-view onset is the first bar": op["first_bar"].mean(),
    "... found in the extended view": op["found"].mean(),
    "... that began in the pre-market": f_["began_pre"].mean(),
    "median move already done at the regular onset (those)": f_.loc[
        f_["began_pre"], "done_by_open"
    ].median(),
    "median regular-hours size (those)": f_.loc[f_["began_pre"], "size"].median(),
}

# --- the close: did the late runs go on after 16:00?
lt = rs.match_runs(
    runs_r[runs_r["at_close"]].reset_index(drop=True), reg.close.index, runs_x, ext.close.index
)
lt["after_close"] = lt["x_peak_ts"].notna() & (
    lt["x_peak_ts"].dt.tz_convert(tz).dt.hour * 60 + lt["x_peak_ts"].dt.tz_convert(tz).dt.minute
    >= 960
)
hit_hour = clock[runs_x["hit"]] / 60
only_after = runs_x[
    (runs_x["part"] == "rth") & (runs_x["hour"] >= 16 - CLOSE_MIN / 60) & (hit_hour >= 16)
]
close_facts = {
    "late runs (share of all regular-hours runs)": len(lt) / len(runs_r),
    "... still running at 16:00 (regular view)": lt["ended_by_close"].mean(),
    "... whose extended run peaked after 16:00": lt["after_close"].mean(),
    "median extra size after the close (those)": (lt["x_size"] - lt["size"])[
        lt["after_close"]
    ].median(),
    "runs starting in the last CLOSE_MIN that reach MOVE_PCT only after 16:00": len(only_after),
}

# --- the evening: after-hours runs against that evening's volume
post = parts == "post"
ah_vol = ext.volume[post].groupby(day_x[post]).sum()
ah_rel = ah_vol / ah_vol.shift(1).rolling(PROFILE_DAYS, min_periods=5).mean()
ev = runs_x[runs_x["part"] == "post"].copy()
ev["day"] = day_x[ev["onset"]]
ev["evening_volume_vs_normal"] = [
    ah_rel.loc[d, s] if d in ah_rel.index else np.nan
    for d, s in zip(ev["day"], ev["symbol"], strict=True)
]
rel_all = ah_rel.to_numpy(float)
rel_all = rel_all[np.isfinite(rel_all)]
evening_facts = {
    "after-hours runs": len(ev),
    "... on an evening with ≥ 3× its normal after-hours volume (news)": (
        ev["evening_volume_vs_normal"] >= 3
    ).mean(),
    "evenings with ≥ 3× normal volume (share of all)": float((rel_all >= 3).mean()),
    "median after-hours run size": ev["size"].median(),
}
display(pd.Series({**open_facts, **close_facts, **evening_facts}, name="value").to_frame().round(3))

fig = make_subplots(
    rows=1,
    cols=3,
    horizontal_spacing=0.07,
    subplot_titles=(
        "2a · run onsets by half hour",
        "2b · open runs: onset in the extended view",
        "2c · after-hours runs vs that evening's volume",
    ),
)
for p, colr in (("pre", plots.SERIES[4]), ("rth", plots.SERIES[0]), ("post", plots.SERIES[2])):
    fig.add_trace(
        go.Histogram(
            x=runs_x.loc[runs_x["part"] == p, "hour"],
            xbins={"start": 4, "end": 20, "size": 0.5},
            name=f"extended view: {p}",
            marker_color=colr,
            opacity=0.75,
        ),
        row=1,
        col=1,
    )
fig.add_trace(
    go.Histogram(
        x=runs_r["hour"],
        xbins={"start": 4, "end": 20, "size": 0.5},
        name="regular view",
        marker={"color": "rgba(0,0,0,0)", "line": {"color": plots._INK, "width": 1.2}},
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Histogram(
        x=f_["x_hour"],
        xbins={"start": 4, "end": 11, "size": 0.25},
        showlegend=False,
        marker_color=plots.SERIES[0],
    ),
    row=1,
    col=2,
)
for k, sym in enumerate(SYMBOLS):
    d = ev[ev["symbol"] == sym]
    fig.add_trace(
        go.Scatter(
            x=d["evening_volume_vs_normal"],
            y=100 * d["size"],
            mode="markers",
            name=sym,
            marker={"color": plots.SERIES[k % len(plots.SERIES)], "size": 6},
        ),
        row=1,
        col=3,
    )
fig.update_layout(barmode="overlay")  # the three parts never share a half hour
fig.add_vline(x=9.5, line={"color": plots._INK, "dash": "dash", "width": 1}, row=1, col=2)
fig.update_xaxes(title_text="hour (New York)", dtick=2, row=1, col=1)
fig.update_xaxes(title_text="hour (New York); dashed = 09:30", row=1, col=2)
fig.update_xaxes(title_text="evening volume ÷ its normal (log)", type="log", row=1, col=3)
fig.update_yaxes(title_text="run size (%)", row=1, col=3)
plots._base(
    fig,
    title=f"Fig 2 · where runs of ≥ {MOVE_PCT:.0%} start, seen with and without the extended hours",
    height=460,
)
below(fig, -0.2)
fig.show()

## 3 · The open, predicted at 9:30 (Fig 3)

One row per stock and day (`research.open_table`). The trade starts at the opening print; the outcome is a race: did
price reach `MOVE_PCT` above or below the open within `RACE_MIN` minutes, and which way first?

* **Regular hours know at 9:30** — the gap (in ATRs), the previous day's return, the last week's return, the ATR,
  yesterday's volume against its average.
* **Only the extended hours know** — last evening's after-hours return and volume; the pre-market's return (≈ the
  gap, seen early), its last `TREND_MIN` minutes, the opening print against the last pre-market print, its volume
  against its own normal and against a regular session, its range in ATRs and where the last print sits in it.

**Table 3** per feature: the rank correlation with the return over the window (t clustered by day), the within-month
AUC for *which way* among the days that moved, and for *whether* it moved (of the feature's size). Then two
logistic models, each month fitted on the months before it only (`research.monthly_walk`): regular-hours inputs
alone against regular + extended. The gain is paired by day (log-loss, t clustered by day).

**Fig 3a** each model's AUC month by month. **Fig 3b / Table 3b** the tradable read — the first-hour return *in the gap's
direction* by fifth of each extended feature (signed toward the gap), days with a gap of at least a quarter ATR,
against one round trip of cost: does the pre-market tell a gap that runs from one that fades?

In [ ]:
ot = rs.open_table(
    ext,
    move_pct=MOVE_PCT,
    window_min=RACE_MIN,
    trend_min=TREND_MIN,
    atr_days=ATR_DAYS,
    profile_days=PROFILE_DAYS,
)
ot["moved"] = (ot["race"] != 0).astype(float)  # NaN race = both barriers in one bar: it moved
movers = ot["race"].isin([1, -1]).to_numpy()
rows = []
for src, cols in (("regular", rs.OPEN_BASE), ("extended", rs.OPEN_EXT)):
    for f in cols:
        x = ot[f].to_numpy(float)
        ic = rs.rank_ic(x, ot["ret"], ot["day"])
        rows.append(
            {
                "feature": f,
                "known from": src,
                "rank IC with the window's return": ic["ic"],
                "t": ic["t"],
                "AUC which way (movers)": rs.monthly_auc(
                    x[movers], (ot.loc[movers, "race"] == 1).astype(int), ot.loc[movers, "month"]
                ),
                "AUC whether (|x|)": rs.monthly_auc(np.abs(x), ot["moved"], ot["month"]),
                "coverage": np.isfinite(x).mean(),
            }
        )
table3 = pd.DataFrame(rows)
print(
    f"{len(ot):,} stock-days · moved {MOVE_PCT:.0%} within {RACE_MIN} min of the open: "
    f"up first {np.mean(ot['race'] == 1):.0%}, down first {np.mean(ot['race'] == -1):.0%}, "
    f"neither {np.mean(ot['race'] == 0):.0%}"
)
display(table3.round(3))

ot = ot.assign(
    abs_gap_atr=ot["gap_atr"].abs(),
    abs_gap=ot["gap"].abs(),
    abs_prev=ot["prev_day_ret"].abs(),
    abs_ah=ot["ah_ret"].abs(),
    abs_trend=ot["pm_trend"].abs(),
    log_pm_share=np.log(ot["pm_vol_share"].clip(lower=1e-4)),
    log_pm_rel=np.log(ot["pm_vol_rel"].clip(lower=1e-3)),
    log_ah_rel=np.log(ot["ah_vol_rel"].clip(lower=1e-3)),
    up=np.where(movers, (ot["race"] == 1).astype(float), np.nan),
)
MODELS = {
    "whether": (
        "moved",
        ["atr_pct", "abs_gap_atr", "abs_gap", "prev_vol_rel", "abs_prev"],
        ["log_pm_share", "log_pm_rel", "pm_range_atr", "abs_ah", "log_ah_rel", "abs_trend"],
    ),
    "which way": (
        "up",
        ["gap_atr", "prev_day_ret", "ret_5d"],
        ["ah_ret", "pm_trend", "open_vs_pm", "pm_range_atr", "pm_pos", "log_pm_rel"],
    ),
}
walk, model_rows = {}, []
for name, (y, base, extra) in MODELS.items():
    d = ot[ot[y].notna()]
    pb, pe = rs.monthly_walk(d, base, y), rs.monthly_walk(d, base + extra, y)
    ok = pb.notna() & pe.notna()
    yy = d.loc[ok, y].to_numpy()

    def logloss(p, yy=yy):
        p = np.clip(p, 1e-6, 1 - 1e-6)
        return -(yy * np.log(p) + (1 - yy) * np.log(1 - p))

    gain = logloss(pb[ok].to_numpy()) - logloss(pe[ok].to_numpy())
    g = rs.day_t(gain, d.loc[ok, "day"])
    walk[name] = (d[ok], pb[ok], pe[ok])
    model_rows.append(
        {
            "question": name,
            "rows": int(ok.sum()),
            "AUC regular only": rs.monthly_auc(pb[ok], yy, d.loc[ok, "month"]),
            "AUC + extended": rs.monthly_auc(pe[ok], yy, d.loc[ok, "month"]),
            "log-loss gain / row": g["mean"],
            "t (by day)": g["t"],
        }
    )
models3 = pd.DataFrame(model_rows).set_index("question")
display(models3.round(4))
d, pb, pe = walk["whether"]
top_rates = {
    lab: d.loc[p >= p.groupby(d["month"]).transform(lambda s: s.quantile(2 / 3)), "moved"].mean()
    for lab, p in (("regular only", pb), ("+ extended", pe))
}
print(
    "days the model ranks in its top third for a move — share that moved: "
    + ", ".join(f"{k} {v:.1%}" for k, v in top_rates.items())
    + f" (all days {d['moved'].mean():.1%})"
)

fig = make_subplots(
    rows=1,
    cols=2,
    horizontal_spacing=0.08,
    subplot_titles=("whether it moves: AUC per month", "which way: AUC per month"),
)
for j, name in enumerate(("whether", "which way"), start=1):
    d, pb, pe = walk[name]
    y = MODELS[name][0]
    for lab, p, colr in (("regular only", pb, plots._MUTED), ("+ extended", pe, plots.SERIES[0])):
        per = d.assign(p=p).groupby("month").apply(lambda g, y=y: rs.auc(g["p"], g[y]))
        fig.add_trace(
            go.Scatter(
                x=list(per.index),
                y=per.to_numpy(),
                mode="lines+markers",
                name=lab,
                showlegend=j == 1,
                line={"color": colr},
            ),
            row=1,
            col=j,
        )
    fig.add_hline(y=0.5, line={"color": plots._INK, "dash": "dot", "width": 1}, row=1, col=j)
plots._base(
    fig,
    title="Fig 3a · the open, month by month: regular-hours inputs alone vs + the extended hours",
    height=400,
)
below(fig, -0.15)
fig.show()

gap_days = ot[ot["gap_atr"].abs() >= 0.25].copy()
gs = np.sign(gap_days["gap"])
toward = {
    "pm_vol_rel": gap_days["pm_vol_rel"],
    "pm_trend →gap": gap_days["pm_trend"] * gs,
    "ah_ret →gap": gap_days["ah_ret"] * gs,
    "open_vs_pm →gap": gap_days["open_vs_pm"] * gs,
    "pm_range_atr": gap_days["pm_range_atr"],
}
cont = gs * gap_days["ret"]
fig = go.Figure()
for k, (f, x) in enumerate(toward.items()):
    fifth = pd.qcut(x, 5, labels=False, duplicates="drop")
    m = 1e4 * cont.groupby(fifth).mean()
    fig.add_trace(
        go.Scatter(
            x=[f"Q{i + 1}" for i in m.index],
            y=m.to_numpy(),
            mode="lines+markers",
            name=f,
            line={"color": plots.SERIES[(k + 2) % len(plots.SERIES)]},
        )
    )
fig.add_trace(
    go.Scatter(
        x=["Q1", "Q5"],
        y=[1e4 * RT] * 2,
        mode="lines",
        name="round-trip cost",
        line={"color": plots._LOSS, "dash": "dot", "width": 1},
    )
)
fig.add_trace(
    go.Scatter(
        x=["Q1", "Q5"],
        y=[0, 0],
        mode="lines",
        showlegend=False,
        line={"color": plots._MUTED, "width": 1},
    )
)
plots._base(
    fig,
    title=f"Fig 3b · gap days: first {RACE_MIN} min toward the gap (bp, before costs), "
    "by fifth of each extended-hours feature",
    height=420,
)
below(fig, -0.15)
fig.show()
table3b = pd.DataFrame(
    [
        {"feature (toward the gap)": f, **rs.rank_ic(x, cont, gap_days["day"])}
        for f, x in toward.items()
    ]
)
print(
    f"gap days (|gap| ≥ 0.25 ATR): {len(gap_days):,}; first {RACE_MIN} min in the gap's direction: "
    f"mean {1e4 * cont.mean():.1f} bp (t {rs.day_t(cont, gap_days['day'])['t']:.2f})"
)
display(table3b.round(3))

## 4 · Confirmations around the clock (Fig 4)

Every confirmation (`research.confirmations`: a close crossing `CONFIRM_PCT` off the lowest low / highest high of the
last `LOOKBACK_MIN` minutes; run = `MOVE_PCT` from that extreme within `WINDOW_MIN`, failed = back through it,
stalled = neither) on both views. The trade enters at the confirmation close and exits at the target, at the
extreme as the stop, or at the last close of the window — before costs.

* In the **regular view** a confirmation at 09:35 can only look back to 09:30; in the **extended view** it sees the
  pre-market's lows and highs. Late in the day the regular view's window ends at 16:00; the extended one runs on.
* A driftless price pays nothing here whatever the stop and target: the break-even is 0 bp before costs, and the
  red line (one regular-hours round trip) is what a trade must clear. Outside regular hours that line is a floor.
* **Read the pre-market bars with Fig 1 in mind.** A confirmation there is often one print at the ask after one at
  the bid; the "failure" that follows is the next print at the bid. On trade prints alone a bid-ask bounce looks
  exactly like a move that reverses — these numbers cannot tell a real fade from the spread.

In [ ]:
kw_conf = dict(
    confirm_pct=CONFIRM_PCT, lookback_min=LOOKBACK_MIN, min_run_pct=MOVE_PCT, max_run_min=WINDOW_MIN
)
cx = rs.confirmations(ext, **kw_conf)
cr = rs.confirmations(reg, **kw_conf)
cx["hour"], cx["day"] = clock[cx["t"]] / 60, day_x[cx["t"]]
cx["resolved_hour"] = clock[cx["end"]] / 60
cx["end_clock"] = clock[cx["end"]]
cr["hour"] = clock_r[cr["t"]] / 60
cr["day"] = np.asarray(reg.close.index.tz_convert(tz).date)[cr["t"]]
EDGES = [4, 7, 8, 9.5, 9.5 + OPEN_MIN / 60, 16 - CLOSE_MIN / 60, 16, 17, 20]
LABELS = [
    "04:00-07:00",
    "07:00-08:00",
    "08:00-09:30",
    "the open",
    "mid-day",
    "the close",
    "16:00-17:00",
    "17:00-20:00",
]


def by_bucket(c, view):
    c = c.assign(bucket=pd.cut(c["hour"], EDGES, right=False, labels=LABELS))
    out = []
    for b, g in c.groupby("bucket", observed=True):
        t_ = rs.day_t(g["ret"], g["day"])
        out.append(
            {
                "view": view,
                "when": b,
                "confirmations": len(g),
                "ran": (g["outcome"] == 1).mean(),
                "failed": (g["outcome"] == -1).mean(),
                "stalled": (g["outcome"] == 0).mean(),
                "mean trade bp": 1e4 * t_["mean"],
                "t (by day)": t_["t"],
                "after one round trip bp": 1e4 * (t_["mean"] - RT),
            }
        )
    return out


table4 = pd.DataFrame(by_bucket(cr, "regular") + by_bucket(cx, "extended"))
display(table4.round(3))

fig = go.Figure()
for view, c, colr, dx in (
    ("regular view", cr, plots._MUTED, -0.04),
    ("extended view", cx, plots.SERIES[0], 0.04),
):
    g = c.assign(h=(c["hour"] * 2).floordiv(1) / 2).groupby("h")
    stats = pd.DataFrame([{"h": h, **rs.day_t(x["ret"], x["day"])} for h, x in g])
    stats = stats[stats["n"] >= 30]
    se = (stats["mean"] / stats["t"]).abs()
    fig.add_trace(
        go.Scatter(
            x=stats["h"] + 0.25 + dx,
            y=1e4 * stats["mean"],
            mode="lines+markers",
            name=view,
            line={"color": colr},
            error_y={
                "type": "data",
                "array": 2e4 * se,
                "visible": True,
                "color": colr,
                "thickness": 1,
            },
        )
    )
fig.add_trace(
    go.Scatter(
        x=[4, 20],
        y=[1e4 * RT] * 2,
        mode="lines",
        name="round-trip cost (regular hours)",
        line={"color": plots._LOSS, "dash": "dot", "width": 1},
    )
)
fig.add_trace(
    go.Scatter(
        x=[4, 20], y=[0, 0], mode="lines", showlegend=False, line={"color": plots._INK, "width": 1}
    )
)
fig.add_vrect(x0=9.5, x1=16, fillcolor=plots._GRID, opacity=0.5, line_width=0, layer="below")
plots._base(
    fig,
    title="Fig 4 · the trade from a confirmation by half hour: mean before costs (bp, ± 2 SE)",
    height=440,
)
below(fig, -0.15)
fig.update_xaxes(title_text="New York time of the confirmation (shaded: regular hours)", dtick=1)
fig.update_yaxes(title_text="bp")
fig.show()

### 4b · Why 08:00 is masked (Table 4b)

The same confirmations on the extended view **as stored** (only the placeholders masked) and **with the 08:00
stale prints masked**. As stored, confirmations made from 04:00 to 08:00 lose 10-33 bp each before costs — a
seemingly strong, every-name, both-sides "pre-market reversal" that even survives entering at the next print.
Most of those "failures" are stopped out in the 08:00 window, by a print from the night before. Masked, the effect
is gone. The lesson for any extended-hours work on these bars: a rule that can be triggered or stopped by one print
must never see the 08:00 minutes.

In [ ]:
ext_s, _ = rs.split_sessions(resample_panel(stored1, BAR_MIN) if BAR_MIN > 1 else stored1)
cs = rs.confirmations(ext_s, **kw_conf)
loc_s = ext_s.close.index.tz_convert(tz)
clock_s = np.asarray(loc_s.hour * 60 + loc_s.minute)
cs["hour"], cs["day"], cs["end_clock"] = (
    clock_s[cs["t"]] / 60,
    np.asarray(loc_s.date)[cs["t"]],
    clock_s[cs["end"]],
)
PRE_EDGES = [4, 6, 7, 8, 9.5]
PRE_LABELS = ["04:00-06:00", "06:00-07:00", "07:00-08:00", "08:00-09:30"]
rows = []
for view, c in (("as stored", cs), ("08:00 masked", cx)):
    c = c[c["hour"] < 9.5].assign(
        bucket=lambda d: pd.cut(d["hour"], PRE_EDGES, right=False, labels=PRE_LABELS)
    )
    for b, g in c.groupby("bucket", observed=True):
        t_ = rs.day_t(g["ret"], g["day"])
        failed = g["outcome"] == -1
        late = failed & g["end_clock"].between(480, 480 + LATE_REPORT_MIN - 1)
        rows.append(
            {
                "view": view,
                "when": b,
                "confirmations": len(g),
                "failed": failed.mean(),
                "mean trade bp": 1e4 * t_["mean"],
                "t (by day)": t_["t"],
                "failures stopped in 08:00-08:09": late.sum() / max(1, failed.sum()),
            }
        )
table4b = pd.DataFrame(rows).set_index(["view", "when"])
display(table4b.round(3))

## 5 · The close (Fig 5)

The confirmations of the last `CLOSE_MIN` minutes of regular hours, **left** with their window cut at 16:00 (the
regular view: a trade still open at the bell is closed there) and **right** allowed to run into the evening. Does
seeing — and trading — the after-hours change what they are worth?

In [ ]:
late_r = cr[(cr["hour"] >= 16 - CLOSE_MIN / 60) & (cr["hour"] < 16)]
late_x = cx[(cx["hour"] >= 16 - CLOSE_MIN / 60) & (cx["hour"] < 16)]
res = pd.DataFrame(
    {
        view: {
            "confirmations": len(c),
            "ran": (c["outcome"] == 1).mean(),
            "failed": (c["outcome"] == -1).mean(),
            "stalled": (c["outcome"] == 0).mean(),
            "mean trade bp": 1e4 * c["ret"].mean(),
            "t (by day)": rs.day_t(c["ret"], c["day"])["t"],
        }
        for view, c in (("cut at 16:00", late_r), ("runs on into the evening", late_x))
    }
)
res.loc["resolved after 16:00", "runs on into the evening"] = (late_x["resolved_hour"] >= 16).mean()
display(res.round(3))
fig = make_subplots(
    rows=1, cols=2, subplot_titles=("cut at 16:00", "runs on into the evening"), shared_yaxes=True
)
for j, c in enumerate((late_r, late_x), start=1):
    for side, sgn, colr in (("up", 1, plots._LONG), ("down", -1, plots._SHORT)):
        d = c[c["direction"] == sgn]
        fig.add_trace(
            go.Histogram(
                x=1e4 * d["ret"],
                xbins={"start": -300, "end": 300, "size": 10},
                name=side,
                showlegend=j == 1,
                marker_color=colr,
                opacity=0.6,
            ),
            row=1,
            col=j,
        )
    fig.add_vline(x=0, line={"color": plots._INK, "dash": "dash", "width": 1}, row=1, col=j)
fig.update_layout(barmode="overlay")
fig.update_xaxes(title_text="trade return (bp, before costs)")
plots._base(
    fig,
    title=f"Fig 5 · confirmations in the last {CLOSE_MIN} minutes: what they earned",
    height=400,
)
below(fig, -0.18)
fig.show()

## 6 · Example days (Figs 6.x)

Each card draws one stock's whole local day, 04:00-20:00 (shaded: regular hours), with its volume. ● marks the onset
the extended view sees and ○ (the larger ring) the one the regular view sees; ★ / ☆ the end of the run in each. `CARDS` days of each kind, drawn
with `SEED`:

* **open runs that began in the pre-market** — the regular view dates them to the open;
* **late runs that went on after the close** — the regular view stops them at 16:00.

In [ ]:
o_, h_, l_, c_, v_ = (
    x.to_numpy(float) for x in (ext.open_, ext.high, ext.low, ext.close, ext.volume)
)
pos_x = pd.Series(np.arange(len(ext.close)), index=ext.close.index)


def day_card(r, kind):
    col = r.col
    d = r.onset_ts.tz_convert(tz).date()
    span = np.flatnonzero(day_x == d)
    x = ext.close.index[span].tz_convert(tz).tz_localize(None)
    colr = plots._LONG if r.direction > 0 else plots._SHORT
    fig = make_subplots(
        rows=2, cols=1, shared_xaxes=True, row_heights=[0.72, 0.28], vertical_spacing=0.04
    )
    fig.add_trace(
        go.Candlestick(
            x=x,
            open=o_[span, col],
            high=h_[span, col],
            low=l_[span, col],
            close=c_[span, col],
            showlegend=False,
            increasing_line_color="#a9c6ba",
            decreasing_line_color="#e0b3b0",
        ),
        row=1,
        col=1,
    )

    def edge(t_):  # the run's extreme at that bar
        return (h_ if r.direction > 0 else l_)[pos_x[t_], col]

    marks = (
        (r.x_onset_ts, r.x_base, "circle", 11, "onset, extended view"),
        (r.onset_ts, r.base, "circle-open", 19, "onset, regular view"),
        (r.x_peak_ts, edge(r.x_peak_ts), "star", 13, "end of the run, extended view"),
        (r.peak_ts, edge(r.peak_ts), "star-open", 19, "end of the run, regular view"),
    )
    for t_, y, sym_, size, name in marks:  # -open symbols are stroked by marker.line
        hollow = sym_.endswith("open")
        fig.add_trace(
            go.Scatter(
                x=[t_.tz_convert(tz).tz_localize(None)],
                y=[y],
                mode="markers",
                name=name,
                marker={
                    "symbol": sym_,
                    "size": size,
                    "color": colr,
                    "line": {"color": colr if hollow else plots._INK, "width": 2 if hollow else 1},
                },
            ),
            row=1,
            col=1,
        )
    fig.add_trace(go.Bar(x=x, y=v_[span, col], name="volume", marker_color="#cfcdc4"), row=2, col=1)
    rth_x = x[parts[span] == "rth"]
    for row in (1, 2):
        fig.add_vrect(
            x0=rth_x[0],
            x1=rth_x[-1],
            fillcolor=plots._GRID,
            opacity=0.45,
            line_width=0,
            layer="below",
            row=row,
            col=1,
        )
    side = "up" if r.direction > 0 else "down"
    title = (
        f"Fig 6 · {r.symbol} {d:%a %Y-%m-%d} — {kind}: {side} run, {r.size:.1%} in regular hours, "
        f"{r.x_size:.1%} in the extended view"
    )
    plots._base(fig, title=title, height=560)
    below(fig, -0.08)
    fig.update_xaxes(rangeslider_visible=False)
    fig.update_yaxes(title_text="price", row=1, col=1)
    fig.update_yaxes(title_text="volume", row=2, col=1)
    return fig


pre_started = op[op["found"] & op["began_pre"]]
went_on = lt[lt["after_close"]]
for kind, pool in (
    ("an open run that began in the pre-market", pre_started),
    ("a late run that went on after the close", went_on),
):
    for r in pool.sample(min(CARDS, len(pool)), random_state=SEED).itertuples():
        day_card(r, kind).show()

## 7 · Checks

In [ ]:
lb = S["_bars"](LOOKBACK_MIN, ext.bar)
causal = all(((c["ref"] <= c["t"]) & (c["t"] - c["ref"] < lb)).all() for c in (cx, cr))
room = all(
    (
        np.where(
            c["direction"] > 0,
            c["entry"] < c["base"] * (1 + MOVE_PCT),
            c["entry"] > c["base"] * (1 - MOVE_PCT),
        )
    ).all()
    for c in (cx, cr)
)
one_day = bool(
    (day_x[runs_x["onset"]] == day_x[runs_x["peak"]]).all()
    and (codes_r[runs_r["onset"]] == codes_r[runs_r["peak"]]).all()
)
no_placeholder = bool(not ((ext1.volume == 0) & ext1.close.notna()).to_numpy().any())
late_masked = bool(
    ext1.close[(minute >= 480) & (minute < 480 + LATE_REPORT_MIN)].isna().all().all()
)
checks = rs.checks_table(
    [
        ("both views hold the same prints", same_prints, True, same_prints),
        (
            f"every {BAR_MIN}-minute candle sits on the 09:30 grid (none straddles the open)",
            grid_ok,
            True,
            grid_ok,
        ),
        ("no placeholder minute carries a price", no_placeholder, True, no_placeholder),
        (
            f"no minute 08:00-08:{LATE_REPORT_MIN - 1:02d} carries a price",
            late_masked,
            True,
            late_masked,
        ),
        ("runs stay inside one day (extended) / one session (regular)", one_day, True, one_day),
        (
            "confirmations only look back (reference ≤ bar, inside LOOKBACK_MIN)",
            causal,
            True,
            causal,
        ),
        ("no confirmation enters past its own target", room, True, room),
        (
            "open runs found in the extended view",
            f"{op['found'].mean():.0%}",
            "≥ 90%",
            op["found"].mean() >= 0.9,
        ),
    ]
)
display(checks)
bucket = table4.set_index(["view", "when"])
rs.scorecard(
    {
        "notebook": "move_extended",
        "symbols": SYMBOLS,
        "move_pct": MOVE_PCT,
        "bar_min": BAR_MIN,
        "reversal_pct": REVERSAL_PCT,
        "confirm_pct": CONFIRM_PCT,
        "lookback_min": LOOKBACK_MIN,
        "open": {k: float(v) for k, v in open_facts.items()},
        "close": {k: float(v) for k, v in close_facts.items()},
        "evening": {k: float(v) for k, v in evening_facts.items()},
        "open_models": {
            q: {
                "auc_regular": models3.loc[q, "AUC regular only"],
                "auc_extended": models3.loc[q, "AUC + extended"],
                "logloss_gain_t": models3.loc[q, "t (by day)"],
            }
            for q in models3.index
        },
        "best_extended_feature_ic_t": float(
            table3.loc[table3["known from"] == "extended", "t"].abs().max()
        ),
        "confirmation_bp": {
            f"{v}|{w}": bucket.loc[(v, w), "mean trade bp"] for v, w in bucket.index
        },
        "premarket_bp_stored_vs_masked": {
            f"{v}|{w}": table4b.loc[(v, w), "mean trade bp"] for v, w in table4b.index
        },
        "late_report_min": LATE_REPORT_MIN,
        "round_trip_bp": 1e4 * RT,
    }
)